# E8b · разбор причины провала замороженного контекста

Дополнение к `e8_cityscapes_colab.ipynb`. Ничего переобучать или пересчитывать E0/kNN не нужно. Используем те же 500 кадров, банк, маски rank/oracle, бюджеты 10%/20%.

**Самый простой запуск:** перенеси четыре кодовые ячейки отсюда в конец работающего ноутбука E8 и выполни по порядку. Если открыл этот файл отдельно в новой VM — сначала восстанови окружение, данные и код ячейками 1–5 основного ноутбука. Основной E8 заново запускать не нужно.

Тестируются только Cityscapes cs_fit/cs_test; отсутствие ACDC в прошлой таблице было намеренным. B0/B5 заморожены. Нужен GPU.

Варианты:
- `entry_raw`: прежний s3 (контроль воспроизводимости).
- `entry`: тот же замороженный контекст, но токены вне выбранных тайлов на выходе stage 3 заменены настоящим выходом полного B5.
- `mid`, `final`: контекст взят из середины / конца настоящей stage 3; выход вне выбранных тайлов также восстановлен.

**Важное ограничение:** восстановление вне тайлов и поздние состояния требуют полного прохода B5. Это диагностическая подсказка, не экономный каскад и не доказательство, что один адаптер решает задачу. Эти состояния воздействуют не только на K/V внимания, но и на пространственные операции блока, включая depthwise-свёртку FFN.


In [ ]:
# 1. Подключение к уже подготовленному запуску, без перезапуска E0/E3/E8.
import os,sys,json,hashlib,subprocess,time
from pathlib import Path
os.environ['USE_TF']='0'
SAVE=Path('/content/drive/MyDrive/e8_cityscapes_v1')
WORK=Path('/content/e8_cityscapes_v1')
EXP=WORK/'experiments'
DATA=WORK/'data'
CACHE=WORK/'cache'
RESULTS=SAVE/'results'
OUT=SAVE/'e8b_v1'
for p in [EXP/'common.py',CACHE/'e8_tiles_cityscapes.npz',RESULTS/'e0_cityscapes.json']:
    assert p.is_file(), f'Нет {p}. Восстанови ячейки 1–5 основного ноутбука E8.'
for domain,n in [('cs_fit',267),('cs_test',233)]:
    assert len(list((CACHE/domain).glob('*.npz')))==n, f'Неполный кэш {domain}'
assert (DATA/'cityscapes/leftImg8bit/val/frankfurt').is_dir(), 'Нужно восстановить распакованные изображения'
OUT.mkdir(exist_ok=True)
os.environ.update(E8_DATA=str(DATA),E8_CACHE=str(CACHE),E8_RESULTS=str(RESULTS),E8B_OUT=str(OUT))
print('Результаты E8b:',OUT)


In [ ]:
# 2. Запись новой версии скрипта. Исходный E8 и его результаты не меняются.
SOURCE='"""E8b: why did E8 fail? Separates two effects and tries other frozen contexts.\n\nE8 had two confounds:\n  1. the frozen context was the stage-entry state, far from what block l expects as keys/values;\n  2. tokens outside the routed tiles stayed at entry values after stage 3, so stage 4 attention and the\n     decode head (bilinear upsampling across tile borders) read them too.\n\nOnly stage 3 (index 2) is masked. B5 runs on the full frame; per image one unmasked pass records the\nreal states: entry (input of block 0), mid (input of block 20), final (output of block 39).\nRouted tokens always start from their real entry state; all other tokens are frozen to one recorded\nstate in the keys/values of every block. Variants:\n  entry_raw - E8\'s "s3": frozen entry state, output outside the routed tiles left as is\n  entry     - same, output outside the routed tiles restored to the real final state (isolates effect 2)\n  mid, final - frozen to a later real state, output restored\nReported as % of the B5-B0 gap closed, plus pixel agreement with full-frame B5 inside routed tiles.\nagree_b0 is the reference: how often B0 itself agrees with B5 on those pixels.\n"""\nimport json\nimport os\nimport time\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom common import (CACHE, DOMAINS, NUM_CLASSES, RESULTS, SIZE, TILE, SegSet, confusion, gain, load,\n                    load_model, miou, tile_sum, top_tiles)\ndef rank(scores):\n    return scores.ravel().argsort().argsort().reshape(scores.shape)\n\nDOMAINS = ["cs_fit", "cs_test"]\nOUT = Path(os.environ["E8B_OUT"])\n\n\nSTAGE = 2\nBUDGETS = [0.1, 0.2]\nVARIANTS = {"entry_raw": ("entry", False), "entry": ("entry", True),\n            "mid": ("mid", True), "final": ("final", True)}\n\n\nclass Frozen(nn.Module):\n    """Wraps block i of n of the masked stage; see the module docstring."""\n\n    def __init__(self, layer, state, i, n):\n        super().__init__()\n        self.layer, self.state, self.i, self.n = layer, state, i, n\n\n    def forward(self, hidden_states, *args, **kwargs):\n        s = self.state\n        if s.recording:\n            if self.i == 0:\n                s.rec["entry"] = hidden_states\n            if self.i == self.n // 2:\n                s.rec["mid"] = hidden_states\n            out = self.layer(hidden_states, *args, **kwargs)\n            if self.i == self.n - 1:\n                s.rec["final"] = out[0]\n            return out\n        if s.mask is None:\n            return self.layer(hidden_states, *args, **kwargs)\n        ctx = s.rec[s.freeze]\n        out = self.layer(torch.where(s.mask, hidden_states, ctx), *args, **kwargs)\n        y = torch.where(s.mask, out[0], ctx)\n        if self.i == self.n - 1 and s.restore:\n            y = torch.where(s.mask, y, s.rec["final"])\n        return (y,) + tuple(out[1:])\n\n\ndef install(model):\n    state = SimpleNamespace(recording=False, rec={}, mask=None, freeze="entry", restore=True)\n    blocks = model.segformer.encoder.block[STAGE]\n    for i, layer in enumerate(blocks):\n        blocks[i] = Frozen(layer, state, i, len(blocks))\n    return state\n\n\ndef token_mask(m):\n    """(8, 16) bool tile mask -> (1, N, 1) bool mask over the tokens of the stage."""\n    k = TILE // (4 * 2 ** STAGE)\n    t = torch.from_numpy(m).cuda().repeat_interleave(k, 0).repeat_interleave(k, 1)\n    return t.reshape(1, -1, 1)\n\n\n@torch.inference_mode()\ndef run(model, state, img, **settings):\n    """Full-frame B5 prediction with the given state settings; the state is reset afterwards."""\n    for k, v in settings.items():\n        setattr(state, k, v)\n    try:\n        logits = model(pixel_values=img).logits\n        return F.interpolate(logits.float(), SIZE, mode="bilinear", align_corners=False).argmax(1)[0].byte().cpu().numpy()\n    finally:\n        state.recording, state.mask = False, None\n\n\ndef sanity(model, state):\n    """The recording pass and an all-true mask (any frozen state) must reproduce the cached B5."""\n    img = SegSet("cs_fit")[0][0][None].cuda().half()\n    (cached,) = next(load("cs_fit", "pred_b5"))\n    agree = (run(model, state, img, recording=True) == cached).mean()\n    assert agree > 0.999, f"recording pass does not reproduce B5, agreement {agree:.4f}"\n    ones = token_mask(np.ones((SIZE[0] // TILE, SIZE[1] // TILE), bool))\n    for freeze in ("entry", "mid", "final"):\n        agree = (run(model, state, img, mask=ones, freeze=freeze, restore=True) == cached).mean()\n        assert agree > 0.999, f"all-true mask ({freeze}) does not reproduce B5, agreement {agree:.4f}"\n\n\ndef main():\n    model = load_model("b5")\n    state = install(model)\n    sanity(model, state)\n\n    base = json.loads((RESULTS / "e0_cityscapes.json").read_text())\n    feats = np.load(CACHE / "e8_tiles_cityscapes.npz")\n    OUT.mkdir(parents=True, exist_ok=True)\n    report_path = OUT / "e8b_cityscapes.json"\n    report = json.loads(report_path.read_text()) if report_path.exists() else {}\n\n\n    print(f"{\'domain\':8} {\'method\':8} {\'budget\':>6} {\'full\':>7}" + "".join(f"{n:>11}" for n in VARIANTS)\n          + "".join(f"{f\'agree {n}\':>16}" for n in VARIANTS) + f"{\'agree_b0\':>10}")\n    for domain in DOMAINS:\n        if domain in report:\n            print(domain, "уже завершён — пропускаем", flush=True)\n            continue\n        dataset = SegSet(domain)\n        assert [p.stem for p, _ in dataset.items] == [p.stem for p in sorted((CACHE / domain).glob("*.npz"))]\n\n        entropy, knn = feats[f"{domain}__entropy__mean"], feats[f"{domain}__knn__mean"]\n        gains = np.stack([tile_sum(gain(*a)) for a in load(domain, "gt", "pred_b0", "pred_b5")])\n        scores = {"rank": rank(entropy) + rank(knn), "oracle": gains}\n        masks = {(k, b): top_tiles(s, b) for k, s in scores.items() for b in BUDGETS}\n\n        zeros = lambda: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)\n        full = {key: zeros() for key in masks}\n        conf = {(*key, n): zeros() for key in masks for n in VARIANTS}\n        agree = {(*key, n): np.zeros(2, np.int64) for key in masks for n in VARIANTS}\n        agree_b0 = {key: np.zeros(2, np.int64) for key in masks}\n\n        per_frame = []\n        started = time.monotonic()\n        loader = DataLoader(dataset, batch_size=None, num_workers=2)\n        for i, ((img, _, _), (gt, b0, b5)) in enumerate(zip(loader, load(domain, "gt", "pred_b0", "pred_b5"))):\n            img = img[None].cuda().half()\n            run(model, state, img, recording=True)\n            for key, m in masks.items():\n                px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n                frame_full = confusion(np.where(px, b5, b0), gt)\n                full[key] += frame_full\n                agree_b0[key] += [(b0 == b5)[px].sum(), px.sum()]\n                tm = token_mask(m[i])\n                for n, (freeze, restore) in VARIANTS.items():\n                    pred = run(model, state, img, mask=tm, freeze=freeze, restore=restore)\n                    frame_conf = confusion(np.where(px, pred, b0), gt)\n                    conf[(*key, n)] += frame_conf\n                    per_frame.append((i, key[0], key[1], n, frame_conf.copy(), frame_full.copy()))\n                    agree[(*key, n)] += [(pred == b5)[px].sum(), px.sum()]\n\n            if (i+1) % 5 == 0 or i+1 == len(dataset):\n                elapsed = time.monotonic()-started\n                eta = elapsed/(i+1)*(len(dataset)-i-1)/60\n                print(f"E8b {domain}: {i+1}/{len(dataset)}, {elapsed/60:.1f} мин, ETA ~{eta:.1f} мин", flush=True)\n\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        closed = lambda c: round(100 * (miou(c) - b0_miou) / (b5_miou - b0_miou), 1)\n        pct = lambda a: round(100 * a[0] / a[1], 1)\n        report[domain] = {}\n        for key in masks:\n            row = {"full": closed(full[key]), "miou_full": float(miou(full[key])),\n                   "miou_b0": b0_miou, "miou_b5": b5_miou}\n            row.update({n: closed(conf[(*key, n)]) for n in VARIANTS})\n            row.update({f"miou_{n}": float(miou(conf[(*key, n)])) for n in VARIANTS})\n            row.update({f"agree_{n}": pct(agree[(*key, n)]) for n in VARIANTS})\n            row["agree_b0"] = pct(agree_b0[key])\n            report[domain][f"{key[0]}@{key[1]}"] = row\n            print(f"{domain:8} {key[0]:8} {key[1]:6.0%} {row[\'full\']:7.1f}"\n                  + "".join(f"{row[n]:11.1f}" for n in VARIANTS)\n                  + "".join(f"{row[f\'agree_{n}\']:16.1f}" for n in VARIANTS) + f"{row[\'agree_b0\']:10.1f}")\n\n        # Save frame confusions for later paired comparisons, before marking domain complete.\n        np.savez_compressed(OUT / f"{domain}_frame_confusions.npz",\n            frame=np.array([r[0] for r in per_frame]),\n            method=np.array([r[1] for r in per_frame]), budget=np.array([r[2] for r in per_frame]),\n            variant=np.array([r[3] for r in per_frame]), conf=np.stack([r[4] for r in per_frame]),\n            full=np.stack([r[5] for r in per_frame]),\n            image_names=np.array([p.stem for p, _ in dataset.items]))\n        tmp=report_path.with_suffix(".json.part")\n        tmp.write_text(json.dumps(report, indent=2));tmp.replace(report_path)\n        print("Сохранён домен:", domain, flush=True)\n\n    print("Результат:", report_path, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
script=EXP/'e8b_cityscapes.py'
source_hash=hashlib.sha256(SOURCE.encode()).hexdigest()
config={'source_sha256':source_hash,'domains':['cs_fit','cs_test'],'budgets':[.1,.2],
        'variants':['entry_raw','entry','mid','final'],'full_b5_restoration':True,
        'cache_signature':json.loads((SAVE/'run_config.json').read_text())}
config_path=OUT/'config.json'
if config_path.exists():
    assert json.loads(config_path.read_text())==config, 'Конфигурация изменилась: выбери новую OUT'
else: config_path.write_text(json.dumps(config,indent=2))
script.write_text(SOURCE)
(OUT/script.name).write_text(SOURCE)
print('Скрипт готов')


In [ ]:
# 3. Запуск E8b. 17 проходов B5 на кадр: запись + 4 выбора × 4 варианта.
# В начале автоматически проверяется совпадение с cached pred_b5.
log_path=OUT/f'e8b_{time.strftime("%Y%m%d_%H%M%S")}.log'
start=time.monotonic()
with log_path.open('w') as log:
    process=subprocess.Popen([sys.executable,'-u',str(script)],cwd=EXP,env=os.environ.copy(),
                             stdout=log,stderr=subprocess.STDOUT)
    try:
        while True:
            try:
                status=process.wait(timeout=30);break
            except subprocess.TimeoutExpired:
                tail=log_path.read_text(errors='replace').splitlines()[-2:]
                print(f'E8b: {(time.monotonic()-start)/60:.1f} мин | '+' | '.join(tail),flush=True)
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
print(log_path.read_text(errors='replace')[-12000:])
assert status==0, f'Ошибка {status}; лог {log_path}'
print('Готово за',round((time.monotonic()-start)/60,1),'мин')


In [ ]:
# 4. Таблица, контроль entry_raw против прежнего s3 и архив.
import csv, zipfile
import pandas as pd
from IPython.display import display
report=json.loads((OUT/'e8b_cityscapes.json').read_text())
rows=[{'domain':d,'method_budget':m,**v} for d,methods in report.items() for m,v in methods.items()]
df=pd.DataFrame(rows)
display(df)
df.to_csv(OUT/'e8b_cityscapes.csv',index=False)
old_path=RESULTS/'e8_cityscapes.json'
if old_path.is_file():
    old=json.loads(old_path.read_text())
    checks=[]
    for d,methods in report.items():
        for m,v in methods.items():
            if d in old and m in old[d]:
                checks.append({'domain':d,'method_budget':m,
                  'delta_entry_raw_vs_s3':round(v['entry_raw']-old[d][m]['s3'],3),
                  'delta_agree':round(v['agree_entry_raw']-old[d][m]['agree_s3'],3),
                  'delta_full':round(v['full']-old[d][m]['full'],3)})
    comparison=pd.DataFrame(checks)
    display(comparison)
    comparison.to_csv(OUT/'control_comparison.csv',index=False)
    if checks and any(abs(c[k])>0.11 for c in checks for k in ['delta_entry_raw_vs_s3','delta_agree','delta_full']):
        print('ВНИМАНИЕ: контроль расходится. Разобрать воспроизводимость до вывода об эффекте восстановления.')
archive=OUT/'e8b_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p!=archive: z.write(p,p.name)
    for name in ['environment.json','e0_cityscapes.json','e8_cityscapes.json','knn_config.json']:
        p=RESULTS/name
        if p.is_file():z.write(p,'e8_reference/'+p.name)
print('Пришли:',archive)
# Чтобы скачать:
# from google.colab import files
# files.download(str(archive))


## Интерпретация

Сначала `entry_raw` должен повторить прежний `s3` на тех же данных и масках. Только после этого сравниваем восстановленные варианты.

Смотрите **mIoU и долю сохранённого эффекта full одновременно с agree**. 95% совпадения — ориентир, а не доказательство полезности: большую долю agreement могут дать одинаковые правильные ответы, тогда как важные исправления потеряются. Сохранены матрицы ошибок по кадрам для последующей парной оценки.

Успех `entry` покажет, что состояние вне тайлов после stage 3 существенно влияет на результат. Успех только `mid/final` укажет на важность согласования контекста с глубиной. Однако эти сравнения не дают чистого разделения всех механизмов: они меняют совместную работу внимания, FFN и последующих слоёв.

Перед E9 нужно решить, как получать и поздний контекст, и восстанавливаемый выход вне регионов без настоящего полного B5. Возможность их предсказания адаптером ещё не проверена.
